In [ ]:
import pandas as pd
import os

# dataset_list = ['banking', 'stackoverflow', 'clinc', 'hwu', 'mcid', 'ele']
dataset_list = ['XTopic']

for dataset in dataset_list:

    data = {}
    for mode in ['train', 'dev', 'test']:
        df = pd.read_csv(f'{dataset}/origin_data/{mode}.tsv', sep='\t')
        df['text'] = df['text'].apply(lambda x: x.replace('\n', ''))
        df['label'] = df['label'].apply(lambda x: x.replace('\n', ''))
        data[mode] = df.drop_duplicates('text')

    train_text = data['train']['text'].tolist()
    dev_text = data['dev']['text'].tolist()

    data['dev'] = data['dev'][data['dev']['text'].apply(lambda x: x not in train_text)]
    data['test'] = data['test'][data['test']['text'].apply(lambda x: x not in train_text + dev_text)]

    for mode in ['train', 'dev', 'test']:
        data[mode].to_csv(f'{dataset}/{mode}.tsv', sep='\t', index=None)


In [1]:
import pandas as pd
import os
from pathlib import Path
import numpy as np
import random

random.seed(42)
np.random.seed(42)

# dataset_list = ['banking', 'stackoverflow', 'clinc', 'hwu', 'mcid', 'ele']
dataset_list = ['XTopic']

for dataset in dataset_list:
    # 1) 读入三份并清洗、去重
    frames = []
    for mode in ['train', 'dev', 'test']:
        df = pd.read_csv(f'{dataset}/origin_data/{mode}.tsv', sep='\t')
        # 防止 NaN
        df['text'] = df['text'].astype(str).fillna('')
        df['label'] = df['label'].astype(str).fillna('')
        # 去掉换行
        df['text'] = df['text'].apply(lambda x: x.replace('\n', ''))
        df['label'] = df['label'].apply(lambda x: x.replace('\n', ''))
        frames.append(df[['text','label']])

    data = pd.concat(frames, ignore_index=True)
    data = data.drop_duplicates(subset='text').reset_index(drop=True)

    # 2) 分层划分：每个 label 至少 1 条测试样本；整体 7:1:2
    groups = data.groupby('label', sort=False)
    train_parts, dev_parts, test_parts = [], [], []

    for lab, g in groups:
        g = g.sample(frac=1.0, random_state=42).reset_index(drop=True)  # 组内打乱
        n = len(g)

        # 基于比例的初始分配
        n_test = max(1, int(n * 0.2))        # 确保至少 1 条进入测试
        n_dev  = int(n * 0.1)
        n_train = n - n_test - n_dev

        # 若样本极少，确保非负
        if n_train < 0:
            # 先保证测试集 1 条，再尽量给 dev，其余留给 train (可能为 0)
            n_test = min(n, max(1, n_test))
            n_dev = max(0, n - n_test - 0)   # train 至少 0
            n_train = n - n_test - n_dev

        # 如果因为四舍五入导致比例偏差太大，做一次微调：保证非负且总和为 n
        if n_train < 0:
            n_train = 0
        leftover = n - (n_train + n_dev + n_test)
        # 将剩余（可能 +/-）优先补到 train，再到 dev
        if leftover > 0:
            take = min(leftover, n - (n_train + n_dev + n_test))
        # 顺序填补
        while leftover > 0:
            n_train += 1
            leftover -= 1
        while leftover < 0:
            # 若超分配，优先从 dev 减，再从 train 减（不动 test，保障至少 1）
            if n_dev > 0:
                n_dev -= 1
                leftover += 1
            elif n_train > 0:
                n_train -= 1
                leftover += 1
            else:
                break  # 极端情况下无法再调

        # 安全校验
        assert n_train >= 0 and n_dev >= 0 and n_test >= 1
        assert n_train + n_dev + n_test == n

        test_parts.append(g.iloc[:n_test])
        dev_parts.append(g.iloc[n_test:n_test+n_dev])
        train_parts.append(g.iloc[n_test+n_dev:])

    train_df = pd.concat(train_parts, ignore_index=True).sample(frac=1.0, random_state=42).reset_index(drop=True)
    dev_df   = pd.concat(dev_parts,   ignore_index=True).sample(frac=1.0, random_state=42).reset_index(drop=True)
    test_df  = pd.concat(test_parts,  ignore_index=True).sample(frac=1.0, random_state=42).reset_index(drop=True)

    # 3) 创建输出目录并保存
    out_dir = Path(dataset) / 'split'
    out_dir.mkdir(parents=True, exist_ok=True)

    train_df.to_csv(out_dir / 'train.tsv', sep='\t', index=False)
    dev_df.to_csv(out_dir / 'dev.tsv', sep='\t', index=False)
    test_df.to_csv(out_dir / 'test.tsv', sep='\t', index=False)

    # 4) 简要统计打印
    print(f'[{dataset}] split done:')
    print('  train/dev/test sizes:', len(train_df), len(dev_df), len(test_df))
    # 验证每个类别的 test 至少 1
    test_counts = test_df['label'].value_counts()
    missing = [lab for lab in data['label'].unique() if test_counts.get(lab, 0) < 1]
    if missing:
        print('  WARNING: some labels missing in test:', missing)
    else:
        print('  OK: every label has at least one test sample.')

[XTopic] split done:
  train/dev/test sizes: 6053 835 1718
  OK: every label has at least one test sample.
